# L15 demo: a datasheet's real token count, a maintenance log's real clusters

Two live activities, matching the module spec.

1. **Tokenize a datasheet-style page** with a real provider tokenizer, count tokens, and
   compare that count against a naive word-count guess, then watch the gap widen on a much
   longer document.
2. **Embed the same 24 maintenance-log entries** used in `figures/lexical-similarity.png`
   with a real hosted embedding model, and check whether "brg vibration increasing over past
   week" finally lands near the other bearing-failure entries, which the TF-IDF baseline in
   the notes could not manage.

**This notebook needs a provider API key and network access.** Set one of:

```bash
export ANTHROPIC_API_KEY=...   # for tokenization via anthropic
export OPENAI_API_KEY=...      # for tokenization via tiktoken, and for embeddings
export VOYAGE_API_KEY=...      # alternative embeddings provider (Anthropic's recommended partner)
```

This session's build sandbox has no route to Anthropic, OpenAI, or Voyage, so the cells
below are written and reviewed but **not executed against a live API** in this repository
(see `figures/make_figures.py` for the offline stand-ins the lecture notes cite instead).
Run this notebook end to end, with a real key, before using it live in class, the same
caveat [L5](../l05/notes.md)'s SECOM download cell carries.


In [ ]:
import os

HAS_ANTHROPIC_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
HAS_OPENAI_KEY = bool(os.environ.get("OPENAI_API_KEY"))
HAS_VOYAGE_KEY = bool(os.environ.get("VOYAGE_API_KEY"))

print(f"Anthropic key present: {HAS_ANTHROPIC_KEY}")
print(f"OpenAI key present:    {HAS_OPENAI_KEY}")
print(f"Voyage key present:    {HAS_VOYAGE_KEY}")

if not (HAS_ANTHROPIC_KEY or HAS_OPENAI_KEY):
    print("\nNo tokenizer-capable key found. Part 1 needs ANTHROPIC_API_KEY or OPENAI_API_KEY.")


## Part 1 — tokenizing a datasheet

The passage below is written in the style of a pump datasheet's cover page: prose,
tolerances, an alloy code, and a couple of unit-bearing values, the same kind of text this
session's notes measured as three to four times more token-dense than ordinary prose.


In [ ]:
DATASHEET_PAGE = """
Model 4820-C centrifugal pump, wetted parts AISI 4140 with an SS316L impeller. Rated
discharge pressure 10.5 MPa at rated flow, casing hydrostatic test at 1.5x rated pressure.
Shaft runout tolerance ±0.05 mm at the coupling face. Suction nozzle diameter Ø25, discharge
nozzle Ø20. Maximum continuous operating temperature 120 C, intermittent excursions to 145 C
are permitted for no more than 15 minutes per event. Bearing lubrication is grease-packed,
relubrication interval 2000 operating hours. Motor frame is NEMA 254T, rated at 15 kW,
service factor 1.15. Refer to the general arrangement drawing for mounting bolt pattern and
torque values; mounting bolts are torqued to 85 N*m in the sequence shown on sheet 2.
""".strip()

naive_word_count = len(DATASHEET_PAGE.split())
print(f"Naive word count: {naive_word_count}")
print(f"Character count (no whitespace): {len(DATASHEET_PAGE.replace(chr(10), '').replace(' ', ''))}")


In [ ]:
def count_tokens_anthropic(text, model="claude-sonnet-4-5"):
    """Real token count via Anthropic's count_tokens endpoint. Needs network + ANTHROPIC_API_KEY.
    Pin the model ID you actually test against; token counts are model-specific."""
    import anthropic

    client = anthropic.Anthropic()
    result = client.messages.count_tokens(
        model=model,
        messages=[{"role": "user", "content": text}],
    )
    return result.input_tokens


def count_tokens_openai(text, model="gpt-4o"):
    """Real token count via tiktoken, OpenAI's own tokenizer library. Needs network on first
    call only, to fetch the encoding for `model`; the encoding itself runs offline after that."""
    import tiktoken

    enc = tiktoken.encoding_for_model(model)
    return len(enc.encode(text))


if HAS_ANTHROPIC_KEY:
    token_count = count_tokens_anthropic(DATASHEET_PAGE)
    provider = "Anthropic"
elif HAS_OPENAI_KEY:
    token_count = count_tokens_openai(DATASHEET_PAGE)
    provider = "OpenAI"
else:
    raise RuntimeError("Set ANTHROPIC_API_KEY or OPENAI_API_KEY to run this cell.")

print(f"{provider} token count: {token_count}")
print(f"Tokens per word (naive): {token_count / naive_word_count:.2f}")


**Never estimate one provider's usage with another provider's tokenizer.** The two functions
above are kept separate on purpose. If you have both keys, run both and compare, they will
not agree, because the vocabularies are trained on different corpora. Use the tokenizer that
belongs to the model you are actually going to call.

Now scale the same page up to see how a naive per-page cost estimate falls apart. Real
datasheets and standards run to dozens of pages; simulate that by repeating this page and
watch the token count grow.


In [ ]:
def count_tokens(text):
    if HAS_ANTHROPIC_KEY:
        return count_tokens_anthropic(text)
    return count_tokens_openai(text)


for n_pages in (1, 5, 10, 40):
    long_doc = (DATASHEET_PAGE + "\n\n") * n_pages
    tokens = count_tokens(long_doc)
    print(f"{n_pages:3d} page(s): {tokens:7d} tokens  ({tokens / n_pages:.0f} tokens/page)")


Pick a context budget for your chosen provider from its **current** documentation (do not
reuse a number from these notes, it will be stale by the time you read this) and find the
page count at which this document would need explicit chunking rather than a single call.
That is the number A8's pipeline needs to check for, per document, before sending anything.


## Part 2 — embedding the maintenance log

The same 24 entries behind `figures/lexical-similarity.png`, so the comparison is apples to
apples: TF-IDF found a real but weak signal (mean within-category cosine similarity 0.130
against 0.009 between categories) and put "brg vibration increasing over past week" at
**zero** similarity to everything, including its own bearing-failure cluster.


In [ ]:
LOG_ENTRIES = [
    ("bearing", "bearing noise reported on drive-end bearing"),
    ("bearing", "noisy bearing, intermittent, worse under load"),
    ("bearing", "brg vibration increasing over past week"),
    ("bearing", "operator heard grinding near the shaft bearing"),
    ("bearing", "abnormal bearing sound at startup, cleared after warmup"),
    ("seal", "seal weeping at the pump gland"),
    ("seal", "leaking gland packing, minor drip rate"),
    ("seal", "pump drips at seal, tightened packing gland"),
    ("seal", "mechanical seal leak, product visible at base"),
    ("seal", "packing gland requires adjustment, slow leak"),
    ("valve", "valve stuck partially open, will not seat"),
    ("valve", "control valve not responding to setpoint change"),
    ("valve", "isolation valve jammed in open position"),
    ("valve", "valve actuator failed to close on command"),
    ("valve", "sticking valve, freed after lubrication"),
    ("corrosion", "corrosion observed on external piping surface"),
    ("corrosion", "rust and pitting on flange face"),
    ("corrosion", "external corrosion under insulation, coating failed"),
    ("corrosion", "surface rust noted on support bracket"),
    ("corrosion", "pitting corrosion found during inspection"),
    ("overheat", "motor running hot, exceeds normal temperature"),
    ("overheat", "high temperature alarm on drive motor"),
    ("overheat", "overheating detected, thermal shutdown triggered"),
    ("overheat", "elevated casing temperature, cooling fan checked"),
]

categories = [c for c, _ in LOG_ENTRIES]
texts = [t for _, t in LOG_ENTRIES]
print(f"{len(texts)} entries across {len(set(categories))} categories")


In [ ]:
def embed_openai(texts, model="text-embedding-3-small"):
    from openai import OpenAI

    client = OpenAI()
    response = client.embeddings.create(model=model, input=texts)
    return [d.embedding for d in response.data]


def embed_voyage(texts, model="voyage-3"):
    import voyageai

    client = voyageai.Client()
    result = client.embed(texts, model=model, input_type="document")
    return result.embeddings


if HAS_OPENAI_KEY:
    embeddings = embed_openai(texts)
    embed_provider = "OpenAI"
elif HAS_VOYAGE_KEY:
    embeddings = embed_voyage(texts)
    embed_provider = "Voyage"
else:
    raise RuntimeError("Set OPENAI_API_KEY or VOYAGE_API_KEY to run this cell.")

print(f"{embed_provider} embeddings: {len(embeddings)} vectors, {len(embeddings[0])} dimensions")


In [ ]:
import numpy as np

X = np.array(embeddings)
X_normed = X / np.linalg.norm(X, axis=1, keepdims=True)
sim = X_normed @ X_normed.T

within, between = [], []
for i in range(len(texts)):
    for j in range(len(texts)):
        if i == j:
            continue
        (within if categories[i] == categories[j] else between).append(sim[i, j])

print(f"Mean within-category cosine similarity: {np.mean(within):.3f}")
print(f"Mean between-category cosine similarity: {np.mean(between):.3f}")

brg_idx = texts.index("brg vibration increasing over past week")
bearing_idxs = [i for i, c in enumerate(categories) if c == "bearing" and i != brg_idx]
print(f"\n'brg vibration' mean similarity to its own (bearing) cluster: "
      f"{np.mean([sim[brg_idx, j] for j in bearing_idxs]):.3f}")
print(f"'brg vibration' best similarity to any other cluster: "
      f"{max(sim[brg_idx, j] for j in range(len(texts)) if categories[j] != 'bearing'):.3f}")

nearest = np.argsort(-sim[brg_idx])[1:4]
print("\n'brg vibration' nearest neighbors:")
for j in nearest:
    print(f"  {sim[brg_idx, j]:.3f}  [{categories[j]}] {texts[j]}")


Compare that last block against the TF-IDF numbers in the notes: mean within/between-category
similarity of 0.130 / 0.009, and exactly 0.000 similarity from "brg vibration" to anything. If
the real embedding model's nearest neighbors for that entry are bearing-category entries, that
is the concrete case for why production retrieval systems use embeddings instead of keyword or
TF-IDF matching, exactly the point L17 builds on for retrieval-augmented
generation.

## What to try next

- Swap in a real datasheet or standard you have access to for Part 1, and find its actual
  page-count-to-context-limit crossover for your chosen provider.
- Add a few more paraphrases with no shared vocabulary to Part 2's log and see whether the
  embedding model still clusters them correctly.
- Log the token usage and estimated cost for both parts, the habit A8 requires from the start.
